In [73]:
# ---------------
# Dependencies
# ---------------

import os
import numpy
import pandas
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

In [74]:
# ------------------
# Reproducibility
# ------------------

def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True,warn_only=True)

seed = 50
set_seed(seed)

In [75]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
# ----------------
# Configuration
# ----------------

DATASET = "/content/drive/MyDrive/data/dataset.csv"
NUM_API_CALLS = 307
SEQUENCE_LENGTH = 100
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [78]:
from sklearn.model_selection import train_test_split

df = pandas.read_csv(DATASET)

X = df.drop(columns=['hash','malware'])
y = df['malware']

def make_balanced(df):
    malware = df[df['malware'] == 1]
    benign = df[df['malware'] == 0]
    malware_down = malware.sample(len(benign), random_state=42)
    return pandas.concat([malware_down, benign]).sample(frac=1, random_state=42)

# Balanced
# df = make_balanced(df)

train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['malware'],
    random_state=42
)

print(train_df)
print(test_df)


                                   hash  t_0  t_1  t_2  t_3  t_4  t_5  t_6  \
35922  d71410ac9cee48f9d57787c40e81c6d3  208  286   76  110  240  117  208   
22047  0e5d2d1d13121c80adb98241fc3fce33  215  274  158  215  274  158  215   
25756  f0dabe473c53a0206692e0f22790e120  215  274  158  215  274  158  215   
33301  4f3b96e7a3f4af165b551fb06329c680  240  117  240  117  240  117  240   
31494  744cd119a11d72765f0688498c75ea66  240  117  240  117  240  117  240   
...                                 ...  ...  ...  ...  ...  ...  ...  ...   
39703  1b41ea7e3f4d7be38365494c9e7b38c2   82  240  117  240  117  240  117   
22263  e981a047aec34a7bd42b98542e8fe45d  112  274  158  215  274  158  215   
23042  2edf17c236ec1ef598b6204cc9728832   82  240  117  240  117  240  117   
1264   36656ceeba7b86f787a43c740539417c   82  240  117  240  117  240  117   
37160  784c9d97d4e0f20558b005ef882064f7   82  240  117  240  117  240  117   

       t_7  t_8  ...  t_91  t_92  t_93  t_94  t_95  t_96  t_97 

In [79]:
# ----------
# Dataset
# ----------

class MalwareGraphDataset(Dataset):
    def __init__(self, df):
        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values, dtype=torch.float32)

    def seq_to_adj(self, seq):
        adj = torch.zeros((NUM_API_CALLS, NUM_API_CALLS))

        for i in range(len(seq)-1):
            src = seq[i]
            dst = seq[i+1]

            if 0 <= src < NUM_API_CALLS and 0 <= dst < NUM_API_CALLS:
                adj[src, dst] = 1

        return adj

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        adj = self.seq_to_adj(seq)

        X = F.one_hot(
            torch.tensor(seq, dtype=torch.long),
            num_classes=NUM_API_CALLS
        ).float().permute(1, 0)

        return adj, X, self.labels[idx]

    def __len__(self):
        return len(self.labels)

In [80]:
# ----------------------------
# Graph Convolutional Layer
# ----------------------------

class GraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(
            torch.randn(in_features, out_features) * 0.01
        )

    def forward(self, adj, X):
        B, N, _ = adj.size()
        I = torch.eye(N, device=adj.device).unsqueeze(0)
        A_hat = adj + I
        D = torch.sum(A_hat, dim=2)
        D_inv = torch.diag_embed(1.0 / (D + 1e-6))
        A_norm = D_inv @ A_hat
        Z = A_norm @ X
        Z = Z @ self.weight
        return Z

In [81]:
# --------
# Model
# --------

class DGCNN(nn.Module):
    def __init__(self, out_channels=31, dropout=0.6):
        super().__init__()
        self.gcn = GraphConvLayer(SEQUENCE_LENGTH, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(NUM_API_CALLS * out_channels, 1)

    def forward(self, adj, X):
        Z = self.gcn(adj, X)
        Z = F.relu(Z)
        Z = self.dropout(Z)
        Z = Z.reshape(Z.size(0), -1)
        out = self.fc(Z)
        return out.squeeze(1)

In [82]:
best_config = [
  0.6,
  128, #32
  10, #30
  31,
]

In [83]:
from sklearn.metrics import roc_auc_score

def train_model(model, train_loader, epochs):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(0.9, 0.999)
    )

    criterion = nn.BCEWithLogitsLoss()

    history = {
        "train_loss": [],
        "train_auc": []
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        all_probs = []
        all_labels = []

        for adj, X, labels in train_loader:
            adj, X, labels = adj.to(DEVICE), X.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(adj, X)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            probs = torch.sigmoid(logits)
            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss /= len(train_loader)
        epoch_auc = roc_auc_score(all_labels, all_probs)

        history["train_loss"].append(epoch_loss)
        history["train_auc"].append(epoch_auc)

        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | AUC: {epoch_auc:.4f}")

    return history

In [84]:
dropout, batch_size, epochs, out_channels = best_config

train_dataset = MalwareGraphDataset(train_df)
test_dataset = MalwareGraphDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

model = DGCNN(out_channels=out_channels, dropout=dropout).to(DEVICE)

history = train_model(model, train_loader, epochs)

Epoch 1/10 | Loss: 0.1348 | AUC: 0.7615
Epoch 2/10 | Loss: 0.0665 | AUC: 0.9216
Epoch 3/10 | Loss: 0.0579 | AUC: 0.9427
Epoch 4/10 | Loss: 0.0541 | AUC: 0.9484
Epoch 5/10 | Loss: 0.0505 | AUC: 0.9573
Epoch 6/10 | Loss: 0.0498 | AUC: 0.9564
Epoch 7/10 | Loss: 0.0473 | AUC: 0.9621
Epoch 8/10 | Loss: 0.0458 | AUC: 0.9641
Epoch 9/10 | Loss: 0.0440 | AUC: 0.9685
Epoch 10/10 | Loss: 0.0432 | AUC: 0.9707


In [85]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

def evaluate(model, test_loader):

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for adj, X, labels in test_loader:
            adj, X = adj.to(DEVICE), X.to(DEVICE)

            logits = model(adj, X)
            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    auc = roc_auc_score(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)

    preds = [1 if x > 0.5 else 0 for x in all_probs]

    print(classification_report(all_labels, preds))
    print(f"PR-AUC: {pr_auc:.4f}")
    print(f"AUC-ROC: {auc:.4f}")

    return all_labels, all_probs

l,p = evaluate(model,test_loader)

              precision    recall  f1-score   support

         0.0       0.85      0.52      0.65       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.92      0.76      0.82     13163
weighted avg       0.98      0.99      0.98     13163

PR-AUC: 0.9987
AUC-ROC: 0.9689


In [86]:
history_df = pandas.DataFrame(history)
history_df.to_csv(f"dgcnn_history_{seed}.csv", index=False)
torch.save(model.state_dict(), f"dgcnn_seed_{seed}.pt")

### Results Compilation
-------
**Seed 10**
```
              precision    recall  f1-score   support

         0.0       0.88      0.55      0.68       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.93      0.77      0.83     13163
weighted avg       0.99      0.99      0.99     13163

PR-AUC: 0.9988
AUC-ROC: 0.9696
```

**Seed 20**
```
              precision    recall  f1-score   support

         0.0       0.88      0.54      0.67       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.93      0.77      0.83     13163
weighted avg       0.99      0.99      0.99     13163

PR-AUC: 0.9987
AUC-ROC: 0.9677
```

**Seed 30**
```
              precision    recall  f1-score   support

         0.0       0.88      0.52      0.66       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.93      0.76      0.82     13163
weighted avg       0.99      0.99      0.98     13163

PR-AUC: 0.9987
AUC-ROC: 0.9681
```

**Seed 40**
```
              precision    recall  f1-score   support

         0.0       0.88      0.55      0.68       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.93      0.77      0.83     13163
weighted avg       0.99      0.99      0.99     13163

PR-AUC: 0.9988
AUC-ROC: 0.9690
```

**Seed 50**
```
              precision    recall  f1-score   support

         0.0       0.85      0.52      0.65       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.92      0.76      0.82     13163
weighted avg       0.98      0.99      0.98     13163

PR-AUC: 0.9987
AUC-ROC: 0.9689
```